# 02613 Mini-Project: Wall Heating!

In [ ]:
from os.path import join
import sys
import numpy as np
import matplotlib.pyplot as plt

### 1. Familiarize yourself with the data. Load and visualize the input data for a few floorplans using a separate Python script, Jupyter notebook or your preferred tool.

In [2]:
def load_data(load_dir, bid):
    SIZE = 512
    u = np.zeros((SIZE + 2, SIZE + 2))
    u[1:-1, 1:-1] = np.load(join(load_dir, f"{bid}_domain.npy"))
    interior_mask = np.load(join(load_dir, f"{bid}_interior.npy"))
    return u, interior_mask

In [3]:
def jacobi(u, interior_mask, max_iter, atol=1e-6):
    u = np.copy(u)

    for i in range(max_iter):
        # Compute average of left, right, up and down neighbors, see eq. (1)
        u_new = 0.25 * (u[1:-1, :-2] + u[1:-1, 2:] + u[:-2, 1:-1] + u[2:, 1:-1])
        u_new_interior = u_new[interior_mask]
        delta = np.abs(u[1:-1, 1:-1][interior_mask] - u_new_interior).max()
        u[1:-1, 1:-1][interior_mask] = u_new_interior

        if delta < atol:
            break
    return u

In [4]:
def summary_stats(u, interior_mask):
    u_interior = u[1:-1, 1:-1][interior_mask]
    mean_temp = u_interior.mean()
    std_temp = u_interior.std()
    pct_above_18 = np.sum(u_interior > 18) / u_interior.size * 100
    pct_below_15 = np.sum(u_interior < 15) / u_interior.size * 100
    return {
        'mean_temp': mean_temp,
        'std_temp': std_temp,
        'pct_above_18': pct_above_18,
        'pct_below_15': pct_below_15,
    }


Load data

In [11]:
LOAD_DIR = '/dtu/projects/02613_2025/data/modified_swiss_dwellings/'
building_ids = [25, 27,49612,73]
N = len(building_ids)
building_ids = building_ids[:N]

# Load floor plans
all_u0 = np.empty((N, 514, 514))
all_interior_mask = np.empty((N, 512, 512), dtype='bool')
for i, bid in enumerate(building_ids):
    u0, interior_mask = load_data(LOAD_DIR, bid)
    all_u0[i] = u0
    all_interior_mask[i] = interior_mask

FileNotFoundError: [Errno 2] No such file or directory: '/dtu/projects/02613_2025/data/modified_swiss_dwellings/25_domain.npy'

Run jacobi iterations for each floor plan

In [ ]:
MAX_ITER = 20_000
ABS_TOL = 1e-4

all_u = np.empty_like(all_u0)
for i, (u0, interior_mask) in enumerate(zip(all_u0, all_interior_mask)):
    u = jacobi(u0, interior_mask, MAX_ITER, ABS_TOL)
    all_u[i] = u

Print summary statistics in CSV format

In [ ]:
stat_keys = ['mean_temp', 'std_temp', 'pct_above_18', 'pct_below_15']
print('building_id, ' + ', '.join(stat_keys))  # CSV header
for bid, u, interior_mask in zip(building_ids, all_u, all_interior_mask):
    stats = summary_stats(u, interior_mask)
    print(f"{bid},", ", ".join(str(stats[k]) for k in stat_keys))

### 2. Familiarize yourself with the provided script. Run and time the reference implementation for a small subset of floorplans (e.g., 10--20). How long do you estimate it would take to process all the floorplans? Perform the timing as a batch job so you get reliable results.

In [9]:
print(f'''The number of the data is {4571}
        real	3m17.537s
        user	3m17.069s
        sys	0m0.185s''')

times = (3*60+17.537)*(4571/20)
print(f"Total estimated time {int(times/60/60)} hr {int(times%3600/60)} min {int(times%60)} sec")

The number of the data is 4571
        real	3m17.537s
        user	3m17.069s
        sys	0m0.185s
Total estimated time 12 hr 32 min 27 sec


### 3. Visualize the simulation results for a few floorplans.

In [ ]:
Vis_N = 4

fig, axes = plt.subplots(3, Vis_N, figsize=(N*5+2, 15))

for i, (id, u0, interior_mask, u) in enumerate(zip(building_ids[:Vis_N], all_u0[:Vis_N], all_interior_mask[:Vis_N], all_u[:Vis_N])):
    title = f"ID: {id}"
    im = axes[0,i].imshow(u0, cmap='magma', aspect='auto')
    axes[0,i].set_title(title)
    axes[1,i].imshow(interior_mask, cmap='binary_r', aspect='auto')  
    im = axes[2,i].imshow(u, cmap='magma', aspect='auto')
cbar_ax = fig.add_axes([0.95, 0.15, 0.02, 0.7])  
fig.colorbar(im, cax=cbar_ax, label="Temperature")

plt.tight_layout(rect=[0, 0, 0.94, 1])  

plt.savefig("Visualize.png")